## 数据初步复制划分

(文件处理脚本)

将原始数据复制到三个文件夹按8:1:1进行分类


```
├── data
│   ├── train(80%)
│   │   ├── SC4001E0-PSG.edf
│   │   ├── SC4001EC-Hypnogram.edf
│   │   └── ...
│   ├── val(10%)
│   │   ├── SC4002E0-PSG.edf
│   │   ├── SC4002EC-Hypnogram.edf
│   │   └── ...
│   ├── test(10%)
│   │   ├── SC40031E0-PSG.edf
│   │   ├── SC40031EC-Hypnogram.edf
│   │   └── ...
```







## 📊 **Sleep-EDF Database Expanded**

 **153 个文件**实际上是 **78 名受试者的数据**（每人有 2 个夜晚的记录）

---

### **文件命名规则**

```
SC4xxxY0-PSG.edf
   |||  |
   |||  └─ Y:记录批次 (E/F/G/H)
   ||└──── 第几晚 (1=第一晚, 2=第二晚)
   |└───── 受试者编号 (001-822)
   └────── Sleep Cassette（数据集代号）

```

---


## 标准检查

R&K 标准

In [3]:
import mne
import os

def check1(name):
    file1 = os.path.join(os.getcwd(), 'sleep-edf', 'sleep-cassette', name)
    annotations = mne.read_annotations(file1)
    print(annotations.description[:5])   # 检查标注描述

def find_hypnogram_files(directory):
    hypnogram_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith("Hypnogram.edf"):
                hypnogram_files.append(file)
    return hypnogram_files

if __name__ == '__main__':
    path1 = os.path.join(os.getcwd(), 'sleep-edf', 'sleep-cassette')
    hypnogram_files = find_hypnogram_files(path1)
    print(hypnogram_files)

    for file in hypnogram_files:
        check1(file)

['SC4001EC-Hypnogram.edf', 'SC4002EC-Hypnogram.edf', 'SC4011EH-Hypnogram.edf', 'SC4012EC-Hypnogram.edf', 'SC4021EH-Hypnogram.edf', 'SC4022EJ-Hypnogram.edf', 'SC4031EC-Hypnogram.edf', 'SC4032EP-Hypnogram.edf', 'SC4041EC-Hypnogram.edf', 'SC4042EC-Hypnogram.edf', 'SC4051EC-Hypnogram.edf', 'SC4052EC-Hypnogram.edf', 'SC4061EC-Hypnogram.edf', 'SC4062EC-Hypnogram.edf', 'SC4071EC-Hypnogram.edf', 'SC4072EH-Hypnogram.edf', 'SC4081EC-Hypnogram.edf', 'SC4082EP-Hypnogram.edf', 'SC4091EC-Hypnogram.edf', 'SC4092EC-Hypnogram.edf', 'SC4101EC-Hypnogram.edf', 'SC4102EC-Hypnogram.edf', 'SC4111EC-Hypnogram.edf', 'SC4112EC-Hypnogram.edf', 'SC4121EC-Hypnogram.edf', 'SC4122EV-Hypnogram.edf', 'SC4131EC-Hypnogram.edf', 'SC4141EU-Hypnogram.edf', 'SC4142EU-Hypnogram.edf', 'SC4151EC-Hypnogram.edf', 'SC4152EC-Hypnogram.edf', 'SC4161EC-Hypnogram.edf', 'SC4162EC-Hypnogram.edf', 'SC4171EU-Hypnogram.edf', 'SC4172EC-Hypnogram.edf', 'SC4181EC-Hypnogram.edf', 'SC4182EC-Hypnogram.edf', 'SC4191EP-Hypnogram.edf', 'SC4192EV-H

## 数据质量控制

检测

In [ ]:
import os
import glob

original_folder = r'./sleep-edf/sleep-cassette'
# --- 1. 扫描和匹配 PSG 和 Hypnogram 文件 ---
# 使用 glob 查找所有以 '-PSG.edf' 结尾的 PSG 文件。
psg_files = glob.glob(os.path.join(original_folder, '*-PSG.edf'))
# 使用 glob 查找所有以 '-Hypnogram.edf' 结尾的 Hypnogram 文件。
hyp_files = glob.glob(os.path.join(original_folder, '*-Hypnogram.edf'))

# 构建一个字典来存储每个患者的 PSG 和 Hypnogram 文件路径，确保它们是配对的。
patient_dict = {}
for psg in psg_files:
    # 从文件名中提取患者ID (例如 'SC4001E0-PSG.edf' -> 'SC4001E0')
    # 这里假设患者ID是文件名前6个字符。需要根据实际文件名格式调整。
    patient_id = os.path.basename(psg)[:6] 
    patient_dict[patient_id] = {'psg': psg, 'hyp': None}  # 初始化 Hypnogram 为 None
for hyp in hyp_files:
    patient_id = os.path.basename(hyp)[:6]
    if patient_id in patient_dict:  # 如果找到对应的 PSG 文件，则更新 Hypnogram 路径
        patient_dict[patient_id]['hyp'] = hyp

# 过滤出所有同时拥有 PSG 和 Hypnogram 文件的有效患者。
valid_patients = [(pid, data['psg'], data['hyp']) for pid, data in patient_dict.items() if data['hyp']]

print((valid_patients))
print(f"Total valid patients: {len(valid_patients)}")  # 打印有效患者数


[('SC4001', './sleep-edf/sleep-cassette\\SC4001E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4001EC-Hypnogram.edf'), ('SC4002', './sleep-edf/sleep-cassette\\SC4002E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4002EC-Hypnogram.edf'), ('SC4011', './sleep-edf/sleep-cassette\\SC4011E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4011EH-Hypnogram.edf'), ('SC4012', './sleep-edf/sleep-cassette\\SC4012E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4012EC-Hypnogram.edf'), ('SC4021', './sleep-edf/sleep-cassette\\SC4021E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4021EH-Hypnogram.edf'), ('SC4022', './sleep-edf/sleep-cassette\\SC4022E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4022EJ-Hypnogram.edf'), ('SC4031', './sleep-edf/sleep-cassette\\SC4031E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4031EC-Hypnogram.edf'), ('SC4032', './sleep-edf/sleep-cassette\\SC4032E0-PSG.edf', './sleep-edf/sleep-cassette\\SC4032EP-Hypnogram.edf'), ('SC4041', './sleep-edf/sleep-cassette\\SC4041E0-PSG.edf', './sleep-edf/sleep-cassette\

### 过程1

尝试对edf文件，使用mne库进行读入psg，一一对应其hypnogram标注文件

处理后数据存储在变量`X_resampled`（展平的特征数据）; `y_resampled` （标签）

最终将处理好的以上俩数据保存至npy

In [ ]:
import os
import shutil
import random
import glob  

# 配置路径
data_folder = r'./sleep-edf/data'  # 处理后的数据文件夹
train_folder = os.path.join(data_folder, 'train')
val_folder = os.path.join(data_folder, 'val')
test_folder = os.path.join(data_folder, 'test')

# 清理和创建子文件夹
for folder in [data_folder, train_folder, val_folder, test_folder]:
    if os.path.exists(folder):
        shutil.rmtree(folder)  # 删除旧的（如果存在）
    os.makedirs(folder)


original_folder = r'./sleep-edf/sleep-cassette'
# --- 1. 扫描和匹配 PSG 和 Hypnogram 文件 ---
# 使用 glob 查找所有以 '-PSG.edf' 结尾的 PSG 文件。
psg_files = glob.glob(os.path.join(original_folder, '*-PSG.edf'))
# 使用 glob 查找所有以 '-Hypnogram.edf' 结尾的 Hypnogram 文件。
hyp_files = glob.glob(os.path.join(original_folder, '*-Hypnogram.edf'))

# 构建一个字典来存储每个患者的 PSG 和 Hypnogram 文件路径，确保它们是配对的。
patient_dict = {}
for psg in psg_files:
    # 从文件名中提取患者ID (例如 'SC4001E0-PSG.edf' -> 'SC4001E0')
    # 这里假设患者ID是文件名前6个字符。需要根据实际文件名格式调整。
    patient_id = os.path.basename(psg)[:6] 
    patient_dict[patient_id] = {'psg': psg, 'hyp': None}  # 初始化 Hypnogram 为 None
for hyp in hyp_files:
    patient_id = os.path.basename(hyp)[:6]
    if patient_id in patient_dict:  # 如果找到对应的 PSG 文件，则更新 Hypnogram 路径
        patient_dict[patient_id]['hyp'] = hyp

# 过滤出所有同时拥有 PSG 和 Hypnogram 文件的有效患者。
valid_patients = [(pid, data['psg'], data['hyp']) for pid, data in patient_dict.items() if data['hyp']]

print(f"Total valid patients: {len(valid_patients)}")  # 打印有效患者数

# --- 2. 划分病人列表 ---
random.seed(42)  # 确保可重复
random.shuffle(valid_patients)  # 随机打乱

total = len(valid_patients)
train_split = int(0.8 * total)
val_split = int(0.9 * total)
train_patients = valid_patients[:train_split]
val_patients = valid_patients[train_split:val_split]
test_patients = valid_patients[val_split:]

print(f"Train patients: {len(train_patients)}, Val patients: {len(val_patients)}, Test patients: {len(test_patients)}")

# --- 3. 复制文件到对应文件夹 ---
def copy_patient_files(patient_list, dest_folder):
    for pid, psg_path, hyp_path in patient_list:
        # 复制 PSG 和 Hypnogram 文件
        shutil.copy(psg_path, os.path.join(dest_folder, os.path.basename(psg_path)))
        shutil.copy(hyp_path, os.path.join(dest_folder, os.path.basename(hyp_path)))
        print(f"Copied patient {pid} to {dest_folder}")  # 可选：打印进度

copy_patient_files(train_patients, train_folder)
copy_patient_files(val_patients, val_folder)
copy_patient_files(test_patients, test_folder)

print("Data split complete! Files copied to train, val, test folders under sleep-cassette-processed.")


Total valid patients: 153
Train patients: 122, Val patients: 15, Test patients: 16
Copied patient SC4722 to ./sleep-edf/data\train
Copied patient SC4701 to ./sleep-edf/data\train
Copied patient SC4802 to ./sleep-edf/data\train
Copied patient SC4201 to ./sleep-edf/data\train
Copied patient SC4341 to ./sleep-edf/data\train
Copied patient SC4251 to ./sleep-edf/data\train
Copied patient SC4641 to ./sleep-edf/data\train
Copied patient SC4762 to ./sleep-edf/data\train
Copied patient SC4522 to ./sleep-edf/data\train
Copied patient SC4351 to ./sleep-edf/data\train
Copied patient SC4811 to ./sleep-edf/data\train
Copied patient SC4091 to ./sleep-edf/data\train
Copied patient SC4672 to ./sleep-edf/data\train
Copied patient SC4621 to ./sleep-edf/data\train
Copied patient SC4501 to ./sleep-edf/data\train
Copied patient SC4001 to ./sleep-edf/data\train
Copied patient SC4702 to ./sleep-edf/data\train
Copied patient SC4212 to ./sleep-edf/data\train
Copied patient SC4371 to ./sleep-edf/data\train
Copie

### 示例调试代码

运行此来检查一个患者的注解

In [ ]:
import mne
patient_id = 'SC4001'  # 替换为实际的出错患者ID
base_path = r'./sleep-edf/sleep-cassette'  # 使用处理后的文件夹路径
stage_dict = {
    'Sleep stage W': 0, # W代表清醒 (Wake)
    'Sleep stage 1': 1, # N1睡眠
    'Sleep stage 2': 2, # N2睡眠
    'Sleep stage 3': 3, # N3睡眠 (深睡眠)
    'Sleep stage 4': 3, # Sleep-EDF原始数据通常有N3和N4，现代标准常将它们合并为N3。
                         # 这里将其与'Sleep stage 3'合并，都映射为标签 3。
    'Sleep stage R': 4, # REM睡眠
    'Sleep stage ?': -1, # 未知或未分类的睡眠阶段。通常会跳过这些数据。
    'W': 0, 'w': 0,      # 数据集中可能存在简化的标注形式 进行扩展
    '1': 1,
    '2': 2,
    '3': 3,
    '4': 3,              # 同样，简化的N4也合并到N3
    'R': 4, 'r': 4,
}

annot_file = os.path.join(base_path, f"{patient_id}EC-Hypnogram.edf")  # 假设文件名结构
psg_file = os.path.join(base_path, f"{patient_id}E0-PSG.edf")
raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
annot = mne.read_annotations(annot_file)
events, event_id = mne.events_from_annotations(raw, stage_dict, verbose=False)
print("Event IDs found:", event_id)
print(events)

# 添加到代码中：就在 events_from_annotations 前
print(f"Patient {patient_id}: Loading annotations...")
print(f"  Unique descriptions: {set(annot.description)}")
print(f"  First 5 desc: {annot.description[:5]}")
print(f"  Raw info: Duration={raw.times[-1]:.1f}s, Channels={raw.ch_names}")
# 然后查看终端输出，这将显示实际注解文本。

# 观察输出，比较与你的 stage_dict active，如果不匹配，请更新字典或报告


C:\Users\Eason\AppData\Local\Temp\ipykernel_46776\2016356641.py:24: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
C:\Users\Eason\AppData\Local\Temp\ipykernel_46776\2016356641.py:24: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
C:\Users\Eason\AppData\Local\Temp\ipykernel_46776\2016356641.py:24: RuntimeWarning: Highpass cutoff frequency 16.0 is greater than lowpass cutoff frequency 0.7, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)


Event IDs found: {'Sleep stage W': 0, 'Sleep stage 1': 1, 'Sleep stage 2': 2, 'Sleep stage 3': 3, 'Sleep stage 4': 3, 'Sleep stage R': 4, 'Sleep stage ?': -1, 'W': 0, 'w': 0, '1': 1, '2': 2, '3': 3, '4': 3, 'R': 4, 'r': 4}
[]
Patient SC4001: Loading annotations...
  Unique descriptions: {np.str_('Sleep stage 3'), np.str_('Sleep stage ?'), np.str_('Sleep stage 2'), np.str_('Sleep stage R'), np.str_('Sleep stage 4'), np.str_('Sleep stage W'), np.str_('Sleep stage 1')}
  First 5 desc: ['Sleep stage W' 'Sleep stage 1' 'Sleep stage 2' 'Sleep stage 3'
 'Sleep stage 2']
  Raw info: Duration=79500.0s, Channels=['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal', 'Resp oro-nasal', 'EMG submental', 'Temp rectal', 'Event marker']


## 质量检测

质量控制标准